# Python Function Arguments

> 📘 **Python Mastery** · Module 03 — Functions · Lesson 2/5

Functions become truly flexible through their arguments: fixed positions, keywords, defaults, and even "give me as many values as you like" with `*args` and `**kwargs`. This lesson covers all of it — including Python's most infamous bug bait, the mutable default argument.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Pass** arguments positionally and explain why order matters
- **Call** functions with keyword arguments in any order you like
- **Write** parameters with sensible default values (and know the ordering rule)
- **Explain** — and avoid — the mutable-default-argument trap
- **Collect** unlimited inputs using `*args` (tuple) and `**kwargs` (dict)
- **Read** signatures containing the `/` and `*` separators (Python 3.8+)

## 1. Positional Arguments

With positional arguments, Python matches values to parameters purely by order: first to first, second to second. Simple and common — but swap two values and Python will not complain; you simply get a wrong answer. Give a function more or fewer arguments than it has parameters, however, and Python stops you with a `TypeError`.

**Syntax:**

```python
def describe_pet(name, species):   # parameter order defines the meaning
    ...

describe_pet("Tommy", "cat")       # Tommy is the name, cat is the species
describe_pet("cat", "Tommy")       # runs fine -- but the meaning is scrambled!
```

**Example:**

In [1]:
def describe_book(title, pages):
    """Describe a book by title and page count."""
    print(f"'{title}' has {pages} pages.")


describe_book("Python Basics", 350)     # correct order
describe_book(350, "Python Basics")     # wrong ORDER -> runs, but nonsense

# Wrong COUNT -> Python raises TypeError
try:
    describe_book("Short Stories")      # missing the 'pages' argument
except TypeError as err:
    print("TypeError:", err)

'Python Basics' has 350 pages.
'350' has Python Basics pages.
TypeError: describe_book() missing 1 required positional argument: 'pages'


## 2. Keyword Arguments

A keyword argument names its target explicitly: `pages=350`. Order stops mattering, and the call becomes self-documenting — anyone reading it knows exactly which value is which.

You can mix positional and keyword arguments, but the positional ones must come **first**.

**Syntax:**

```python
def function(a, b, c): ...

function(c=3, a=1, b=2)      # any order, all named
function(1, c=3, b=2)        # positional first, keywords after
function(a=1, 2, 3)          # SyntaxError: positional argument follows keyword
```

**Example:**

In [2]:
def book_ticket(movie, seats, show_time):
    """Print a cinema booking summary."""
    print(f"{seats} seat(s) for '{movie}' at {show_time}.")


book_ticket("Interstellar", 3, "7:30 PM")    # all positional

# All keywords: order is completely free
book_ticket(show_time="7:30 PM", movie="Interstellar", seats=3)

# Mixed: positional arguments first, keywords after
book_ticket("Inception", seats=2, show_time="9:00 PM")

# Keywords FIRST would be a SyntaxError (so it is shown, not run):
# book_ticket(seats=2, "Inception", show_time="9:00 PM")

3 seat(s) for 'Interstellar' at 7:30 PM.
3 seat(s) for 'Interstellar' at 7:30 PM.
2 seat(s) for 'Inception' at 9:00 PM.


## 3. Default Parameter Values

A default value makes a parameter optional: callers who omit it receive the fallback. This keeps everyday calls short while still allowing full control.

One hard rule: **once a parameter has a default, every parameter after it must have one too.** Otherwise Python could not tell which supplied value belongs to a later parameter without a default.

**Syntax:**

```python
def function(required, optional=value): ...   # optional comes AFTER required

# Non-default AFTER default -> SyntaxError:
# def broken(a=1, b): ...
```

**Example:**

In [3]:
def make_tea(flavor="milk", sugar_level=1):
    """Prepare tea with sensible defaults."""
    return f"One cup of {flavor} tea with {sugar_level} spoon(s) of sugar."


print(make_tea())                  # use BOTH defaults
print(make_tea("green"))           # override the first only
print(make_tea("lemon", 0))        # override both
print(make_tea(sugar_level=2))     # override by keyword, skipping the first

One cup of milk tea with 1 spoon(s) of sugar.
One cup of green tea with 1 spoon(s) of sugar.
One cup of lemon tea with 0 spoon(s) of sugar.
One cup of milk tea with 2 spoon(s) of sugar.


In [4]:
# Python enforces the rule BEFORE running anything -- watch the parser object:
source_code = """
def broken_tea(sugar=1, flavor):
    return flavor
"""

try:
    compile(source_code, "<demo>", "exec")
except SyntaxError as err:
    print("SyntaxError:", err.msg)

SyntaxError: parameter without a default follows parameter with a default


## 4. ⚠️ The Mutable Default Gotcha

This is the classic Python trap. Default values are created **once**, when the `def` line runs — not freshly on every call. If that default is a mutable object like a list or a dict, every call that uses the default **shares that very same object**.

So `add_item("apple")` looks like it starts a fresh cart each time. It does not: items pile up across calls like forgotten groceries.

> 🔍 **Under the Hood:** defaults live in a tuple attached to the function object itself — visible as `add_item.__defaults__`. The list is constructed once, at definition time, and survives between calls because there is literally only one list.

**Syntax:**

```python
# BUGGY: one shared list across ALL calls
def add_item(item, cart=[]):
    cart.append(item)
    return cart

# FIXED: a None sentinel creates a fresh list per call
def add_item(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart
```

**Example:** watch the bug happen, then prove why.

In [5]:
def add_item_buggy(item, cart=[]):
    cart.append(item)
    return cart


print(add_item_buggy("apples"))     # ['apples'] -- looks fine
print(add_item_buggy("milk"))       # ['apples', 'milk'] -- surprise!
print(add_item_buggy("eggs"))       # ['apples', 'milk', 'eggs']

# Proof: ONE list is stored INSIDE the function object between calls
print("Stored defaults:", add_item_buggy.__defaults__)

['apples']
['apples', 'milk']
['apples', 'milk', 'eggs']
Stored defaults: (['apples', 'milk', 'eggs'],)


In [6]:
def add_item_safe(item, cart=None):
    """Append item to cart, creating a fresh list whenever none is given."""
    if cart is None:
        cart = []                    # brand-new list on EVERY call that needs it
    cart.append(item)
    return cart


cart_sarah = add_item_safe("apples")
print(cart_sarah)                            # ['apples']
print(add_item_safe("milk"))                 # ['milk'] -- fresh list, as expected
print(add_item_safe("rice", cart_sarah))     # passing YOUR OWN list still works
print(cart_sarah)                            # ['apples', 'rice']

['apples']
['milk']
['apples', 'rice']
['apples', 'rice']


## 5. Arbitrary Positional Arguments: `*args`

Sometimes you cannot know in advance how many values a caller will send — totalling prices, joining names. Prefixing a parameter with `*` collects any number of extra positional arguments into a **tuple**.

The star does the magic; the name `args` is mere convention. Inside the function it behaves like an ordinary tuple.

**Syntax:**

```python
def function(*args):            # args becomes a TUPLE of extras
    ...

function(1, 2, 3, 4)            # inside: args == (1, 2, 3, 4)
```

**Example:**

In [7]:
def total_bill(*prices):
    """Add up any number of menu prices."""
    print("Received:", prices, "| type:", type(prices).__name__)
    return sum(prices)


print(total_bill(120))                 # one item
print(total_bill(120, 45, 300, 80))    # four items
print(total_bill())                    # zero items -> empty tuple, sum 0


# Regular parameters come FIRST, *args mops up the rest
def order_summary(table, *dishes):
    return f"Table {table} ordered {len(dishes)} dish(es): {', '.join(dishes)}"


print(order_summary(4, "biriyani", "borhani", "falooda"))

Received: (120,) | type: tuple
120
Received: (120, 45, 300, 80) | type: tuple
545
Received: () | type: tuple
0
Table 4 ordered 3 dish(es): biriyani, borhani, falooda


In [8]:
def total_bill(*prices):
    return sum(prices)


menu_prices = [120, 45, 300]

# The star also works at the CALL site: unpack a sequence into arguments
print(total_bill(*menu_prices))        # same as total_bill(120, 45, 300)


def charge(amount, currency="taka"):
    return f"Charged {amount} {currency}"


order = {"amount": 35}
print(charge(**order))                 # ** unpacks a dict into keywords

465
Charged 35 taka


## 6. Arbitrary Keyword Arguments: `**kwargs`

Two stars collect any number of extra **keyword** arguments into a **dictionary**: each keyword becomes a key, its value the value. Perfect for optional settings you want to inspect by name or forward elsewhere.

Again, the stars are the syntax; `kwargs` ("keyword arguments") is just the traditional spelling.

**Syntax:**

```python
def function(**kwargs):           # kwargs becomes a DICT
    ...

function(color="red", size=10)    # inside: kwargs == {'color': 'red', 'size': 10}
```

**Example:**

In [9]:
def build_profile(name, **details):
    """Build a student profile from any number of keyword details."""
    print("Extras received:", details, "| type:", type(details).__name__)
    profile = {"name": name}
    profile.update(details)
    return profile


sarah = build_profile("Sarah", age=21, city="Dhaka", course="Python")
print(sarah)

for key, value in sarah.items():
    print(f"  {key}: {value}")

Extras received: {'age': 21, 'city': 'Dhaka', 'course': 'Python'} | type: dict
{'name': 'Sarah', 'age': 21, 'city': 'Dhaka', 'course': 'Python'}
  name: Sarah
  age: 21
  city: Dhaka
  course: Python


In [10]:
# The wrapper pattern: accept **kwargs just to FORWARD them
def format_event(event, **details):
    pairs = ", ".join(f"{k}={v}" for k, v in details.items())
    return f"[{event}] {pairs}"


def log_event(event, **details):
    """Log an event, passing any keyword details through untouched."""
    return format_event(event, **details)


print(log_event("login", user="sarah", ip="10.0.0.5"))
print(log_event("purchase", item="novel", price=450))

[login] user=sarah, ip=10.0.0.5
[purchase] item=novel, price=450


## 7. Putting It All Together: Parameter Order

A signature may combine every kind of parameter, but Python enforces one exact order:

**positional → `*args` → keyword-with-default → `**kwargs`**

Read any well-written signature left to right and you will find this shape. Get it wrong and it is a `SyntaxError` before the program even starts.

**Syntax:**

```python
def function(pos1, pos2, *args, kw1=default, kw2=default2, **kwargs):
    ...
```

**Example:**

In [11]:
def place_order(customer, *items, priority=False, **extras):
    """Full signature: positional, *args, keyword default, **kwargs."""
    summary = f"Order for {customer}: {len(items)} item(s)"
    if priority:
        summary += " [PRIORITY]"
    if extras:
        summary += f" | extras={extras}"
    return summary


print(place_order("Sarah", "pen", "notebook"))
print(place_order("Rahim", "calculator", priority=True))
print(place_order("Aisha", "bag", gift_wrap=True, note="Happy birthday"))

# Breaking the order rule -> SyntaxError, caught before any code runs:
bad_signature = """
def oops(**kwargs, *args):
    pass
"""
try:
    compile(bad_signature, "<demo>", "exec")
except SyntaxError as err:
    print("SyntaxError:", err.msg)

Order for Sarah: 2 item(s)
Order for Rahim: 1 item(s) [PRIORITY]
Order for Aisha: 1 item(s) | extras={'gift_wrap': True, 'note': 'Happy birthday'}
SyntaxError: arguments cannot follow var-keyword argument


## 8. `/` and `*` as Pure Separators (Python 3.8+)

Besides collecting values, `/` and `*` can appear **alone** in a signature as boundary markers:

- Everything **left of `/`** is *positional-only*: it can never be passed by name.
- Everything **right of `*`** is *keyword-only*: it can never be filled by position.

The standard library leans on this heavily — for instance `sorted(iterable, /, *, key=None, reverse=False)`: give the iterable by position, but spell out `key=` and `reverse=` by name.

**Syntax:**

```python
def function(a, b, /, c, *, d):   # a, b positional-only; d keyword-only
    ...

function(1, 2, 3, d=4)            # the only way to call it
```

**Example:**

In [12]:
def exam_report(name, roll, /, *, grade):
    """'name' and 'roll' are positional-only; 'grade' is keyword-only."""
    return f"Roll {roll}, {name}, scored grade {grade}"


print(exam_report("Sarah", 12, grade="A"))     # the intended style

# Passing a positional-only parameter by name fails:
try:
    exam_report(name="Sarah", roll=12, grade="A")
except TypeError as err:
    print("TypeError:", err)

# Filling a keyword-only parameter by position fails too:
try:
    exam_report("Rahim", 7, "B")
except TypeError as err:
    print("TypeError:", err)

# Real-world taste: this is exactly sorted()'s signature in action
nums = [3, 1, 2]
print(sorted(nums, key=None, reverse=True))

Roll 12, Sarah, scored grade A
TypeError: exam_report() got some positional-only arguments passed as keyword arguments: 'name, roll'
TypeError: exam_report() takes 2 positional arguments but 3 were given
[3, 2, 1]


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| --- | --- | --- |
| `def f(item, lst=[])` (or `{}`) | One shared mutable object mutated across every call | Use a `None` sentinel and create the container inside |
| `f(x=1, 2)` — keyword before positional | `SyntaxError` at parse time; nothing runs | Positional arguments first, keywords after |
| Treating `args` in `*args` as a list | It is a **tuple** — `.append()` and item assignment fail | Convert when needed: `items = list(args)` |
| Writing `**kwargs` before `*args` or defaults | `SyntaxError`: parameter order violated | Remember: positional → `*args` → defaults → `**kwargs` |
| Routing core options through `**kwargs` | Typos vanish silently: `f(nmae="X")` just lands in the dict | Keep essential parameters explicit; reserve `**kwargs` for true extras |

## 💡 Best Practices & Pro Tips

- Default to immutable defaults (`None`, `0`, `""`). Reach for a `None` sentinel the moment a default would be a list, dict, or set.
- At call sites, spell flags out as keywords: `plot_series(data, smooth=True)` beats `plot_series(data, True)`.
- Use `*args`/`**kwargs` mainly in thin wrappers that *forward* arguments (loggers, decorators, plotting helpers) — not to hide a fuzzy API.
- More than about four parameters? Group related ones or pass a config dict/object.
- 🤖 **AI-engineering relevance:** scientific Python is built on this vocabulary. `plt.plot(x, y, color="red", linewidth=2)` passes style options as keywords; scikit-learn estimators take dozens of defaulted keyword arguments; NumPy signatures like `np.all(a, /, axis=None, out=None, keepdims=False)` use `/` and keyword-only markers. Being able to *read* such signatures is a daily skill.

## 📌 Summary

| Feature | What it does | Example |
| --- | --- | --- |
| `def f(a, b)` | Plain positional parameters | `f(1, 2)` |
| `f(a=1, b=2)` | Keyword arguments — any order, self-documenting | `book_ticket(seats=3, movie="Up", show_time="6pm")` |
| `def f(x=0)` | Default value makes the parameter optional | `make_tea("green")` |
| `def f(*args)` | Extra positionals collected into a **tuple** | `total_bill(120, 45)` |
| `def f(**kwargs)` | Extra keywords collected into a **dict** | `build_profile("Sarah", age=21)` |
| `f(*seq)` / `f(**dict)` | Unpack at the call site | `charge(**{"amount": 35})` |
| `def f(a, /)` | Parameters before `/` are positional-only | `sorted(nums)` |
| `def f(*, key)` | Parameters after `*` are keyword-only | `sorted(nums, key=len)` |
| `f.__defaults__` | Tuple of stored defaults (mutable-gotcha evidence) | `add_item.__defaults__` |

Key takeaways:

- Position is a contract; keywords make calls self-documenting and order-free.
- Defaults are evaluated **once, at `def` time** — mutable defaults are shared state in disguise.
- `*args` packs leftovers into a tuple; `**kwargs` packs them into a dict.
- Signature order is law: positional → `*args` → keyword/default → `**kwargs`.

## 🔗 Next Lesson

➡️ Continue with **03_Return_Values** — what comes back out of a function: single values, tuples, early exits, and the classic print-vs-return bug.